# EOA - Protocolo Experimental em Notebook

Este *notebook* conduz a avaliação prática do framework **EOA (*Explanation-Oriented Agents*)** sobre os dados reais do [sistema hidráulico da ZeMA](https://zenodo.org/records/1323611).

- **Parte 1 (Demonstração Guiada):** roteiro direto contemplando inicialização com persistência em SQLite (`eoa.db`), registro inicial de duas ferramentas de acesso a dados, consulta apenas com documentação do sistema, listagem de incidentes, explicação de uma predição e criação, ajuste e remoção de um agente especialista.
- **Parte 2 (Experimentação Livre):** espaço aberto para exploração livre do framework, incluindo as duas ferramentas de telemetria e comparação estatística que permanecem disponíveis em [`tools.py`](./tools.py).

> Consulte [`GUIDE.md`](./GUIDE.md) sempre que precisar conferir parâmetros, contratos e métodos completos.

---
## Parte 1: Demonstração Guiada

### 1. Configuração do ambiente
Carrega a chave da API do Gemini a partir do arquivo `.env` e define uma função auxiliar para exibir a resposta explicativa, a nota de validação e o rastro de execução dos agentes.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Carrega a chave de API definida no arquivo .env deste diretório
load_dotenv(Path(".env"))
print(f"GEMINI_API_KEY carregada: {bool(os.environ.get('GEMINI_API_KEY'))}")


def mostrar_resposta(res) -> None:
    """Exibe a narrativa gerada, a pontuação do validador e o rastro dos agentes."""
    print("=" * 80)
    print("EXPLICAÇÃO GERADA:")
    print("-" * 80)
    print(res.narrative)
    print("-" * 80)
    print(
        f"Nota Validador: {res.validation_score}/10.0 | "
        f"Aprovado: {res.validation_passed} | "
        f"Tentativas: {res.attempts_used} | "
        f"Fallback: {res.fallback_used}"
    )
    print("-" * 80)
    for step in res.execution_trace:
        print(f"[{step.agent_name:<23}] {step.status:<10} | {step.execution_time_ms:7.1f} ms | {step.summary}")
    print("=" * 80)

### 2. Importações e inicialização com persistência em SQLite (`eoa.db`)
Importam-se as classes principais do EOA e as funções auxiliares de [`tools.py`](./tools.py). Em seguida, cria-se o armazenamento local `SQLiteSessionStore("eoa.db")`, define-se um identificador único de sessão (`SESSION_ID`) e inicializa-se a camada `eoa` via construtor `ExplanationLayer`.

In [ ]:
from eoa import ExplanationLayer, SpecializedAgent, SQLiteSessionStore
from tools import (
    get_baseline_comparison,
    get_incident,
    get_sensor_readings,
    list_incidents,
    load_domain_context,
)

# Banco SQLite local ('eoa.db') para registrar sessões, histórico e auditorias de validação
store = SQLiteSessionStore("eoa.db")
SESSION_ID = "sessao_cooler"

# Inicializa a camada pelo construtor direto utilizando gemini-3.5-flash-lite.
# O parâmetro auto_create_agents=True instancia automaticamente os quatro agentes padrão.
eoa = ExplanationLayer(
    provider="gemini",
    model="gemini-3.5-flash-lite",
    session_store=store,
    auto_create_agents=True,
)
eoa.clear_session(SESSION_ID)
eoa.inspect()

### 3. Registro da ferramenta de contexto no agente responsável (`context_manager`)
Em vez de fornecer o contexto manualmente em cada chamada, registra-se `load_domain_context` no agente de contexto (`context_manager`). As funções `get_sensor_readings` e `get_baseline_comparison` permanecem livres para experimentação na Parte 2.

In [ ]:
context_agent = eoa.get_agent("context_manager")

# load_domain_context(): carrega a documentação técnica da ZeMA, os 14 sensores e limiares saudáveis
context_agent.register_tool(load_domain_context)

# Confira na coluna 'Tools' que o context_manager possui 1 ferramenta registrada
eoa.inspect()

### 4. Consulta sem modelo preditivo (usando apenas documentação do domínio)
Executa-se `eoa.explain(...)` passando apenas a pergunta e o `SESSION_ID`, sem saída preditiva. O agente de contexto consulta `load_domain_context` automaticamente, enquanto o agente de artefatos (`artifacts`) identifica que não há predição a avaliar e omite essa etapa (`0.0 ms`).

In [ ]:
res_contexto = eoa.explain(
    session_id=SESSION_ID,
    user_query=(
        "Explique em termos práticos como funciona o sistema hidráulico ZeMA, "
        "quais são os grupos de sensores físicos monitorados e o que significam "
        "os três códigos de condição do refrigerador (Cooler condition / %)."
    ),
)

mostrar_resposta(res_contexto)

### 5. Consulta aos incidentes disponíveis com `list_incidents`
Antes de solicitar a explicação de uma predição específica, utiliza-se `list_incidents()` para inspecionar os nove incidentes representativos pré-processados com `RandomForestClassifier` e valores TreeSHAP.

In [ ]:
# Funções disponíveis em tools.py para uso direto ou acoplamento em agentes:
# - list_incidents()                     -> lista os 9 incidentes selecionados (códigos 3, 20 e 100)
# - list_incidents(condition_code=3)     -> filtra incidentes por código de eficiência (3, 20 ou 100)
# - load_domain_context()                -> carrega documentação técnica da ZeMA e referência dos sensores
# - get_incident(incident_id)            -> retorna {'prediction': ..., 'xai': ...} do incidente especificado
# - get_sensor_readings(incident_id)     -> retorna as médias dos 14 sensores físicos e stable_flag
# - get_baseline_comparison(incident_id) -> retorna delta, z-score e percentis vs. operação saudável (100%)

incidentes = list_incidents()

print(f"{'ID':<4} {'Ciclo':<7} {'Condição Prevista':<30} {'Conf.':<7} {'Stable':<7} {'Top 3 TreeSHAP'}")
print("-" * 100)
for inc in incidentes:
    top3 = ", ".join(f"{k}: {v:+.4f}" for k, v in inc["top_3_shap_features"].items())
    print(
        f"{inc['incident_id']:<4} {inc['cycle_id']:<7} "
        f"{inc['predicted_condition_label']:<30} {inc['confidence']:<7.2f} "
        f"{inc['stable_flag']:<7} {top3}"
    )

### 6. Explicação de uma predição a partir do identificador do incidente
Registra-se `get_incident` no agente de artefatos (`artifacts`) e solicita-se a explicação do `incident_id=5` (eficiência reduzida em `20%`, com confiança moderada de `0.58`) na mesma sessão `SESSION_ID`. O framework recupera automaticamente a predição, as atribuições TreeSHAP e as regras do domínio.

In [ ]:
# Vincula a ferramenta de consulta de predição e SHAP ao agente de artefatos
eoa.get_agent("artifacts").register_tool(get_incident)

res_predicao = eoa.explain(
    incident_id=5,
    session_id=SESSION_ID,
    user_query=(
        "Por que o modelo classificou o incidente 5 como 20% (eficiência reduzida)? "
        "Relacione os principais atributos do TreeSHAP com as regras e limiares do domínio."
    ),
)

mostrar_resposta(res_predicao)

### 7. Criação de agente especialista, sobrescrita de prompt, execução e remoção
Demonstra-se a extensibilidade dinâmica na mesma sessão (`SESSION_ID`) e incidente (`incident_id=5`): cria-se um `SpecializedAgent` equipado com `get_sensor_readings` para investigar a confiança moderada e verificar o regime térmico (`stable_flag`), sobrescrevem-se suas instruções via `override_agent_prompt`, executa-se a consulta e remove-se o agente ao final com `remove_agent`.

In [ ]:
# 1. Cria um agente especialista (na v0.1.0 o construtor exige um prompt inicial não vazio).
#    Passamos get_sensor_readings em tools=[] para consultar stable_flag e TS1-TS4 em paralelo.
agente_regime = SpecializedAgent(
    name="analista_regime_termico",
    role="Especialista em Estabilidade Térmica",
    prompt="Analise o incidente.",
    tools=[get_sensor_readings],
)
eoa.register_agent(agente_regime)

# 2. Sobrescreve o prompt do agente usando os dados obtidos pela ferramenta ({tool_outputs})
eoa.override_agent_prompt(
    "analista_regime_termico",
    """Você é um Especialista em Estabilidade Térmica de Sistemas Hidráulicos.
Analise os dados de telemetria coletados ({tool_outputs}).
Em no máximo 3 linhas objetivas:
1. Informe se o ciclo está em regime estável (stable_flag=0) ou em transição (stable_flag=1).
2. Explique se os sensores de temperatura (TS1 a TS4) confirmam aquecimento consistente ou fase transitória.
""",
)

# Confira o novo agente e a origem do prompt ('Inline (str)') na tabela
eoa.inspect()

# 3. Continua a investigação do incidente 5 na mesma sessão com o especialista ativo
res_especialista = eoa.explain(
    incident_id=5,
    session_id=SESSION_ID,
    user_query="Considerando a estabilidade térmica desse ciclo, os sensores de temperatura reforçam ou explicam a confiança de 58% no incidente 5?",
)
mostrar_resposta(res_especialista)

# 4. Remove o agente especialista ao final da demonstração e verifica a topologia restaurada
eoa.remove_agent("analista_regime_termico")
eoa.inspect()

# Confirma a persistência da sessão no banco SQLite 'eoa.db'
print("Sessões salvas em eoa.db:", store.list_sessions())

---
## Parte 2: Experimentação Livre

A célula abaixo constitui um espaço aberto para exploração livre do framework EOA, podendo ser editada ou expandida com novas células conforme o interesse do participante.

Sugestões de exploração com os recursos já configurados neste ambiente:
- **Registrar as duas ferramentas restantes de [`tools.py`](./tools.py):** adicionar `get_sensor_readings` e `get_baseline_comparison` ao `context_manager` (`eoa.get_agent("context_manager").register_tool(...)`) e observar como a explicação passa a incorporar leituras exatas dos sensores, desvios (`delta`, `z_score`) e percentis frente à operação saudável.
- **Criar e registrar novos agentes especialistas:** instanciar `SpecializedAgent(...)` com foco específico (por exemplo, análise de pressão `PS1` a `PS6` ou vazão `FS1` e `FS2`) e registrar via `eoa.register_agent(...)`.
- **Refinar instruções (*prompts*):** testar `eoa.override_agent_prompt("explainer", ...)` ou restaurar a instrução padrão com `eoa.reset_agent_prompt("explainer")`.
- **Explorar outros incidentes e histórico:** testar um novo `session_id` para outros incidentes (`1` a `9`), fornecer `prediction=` / `xai=` diretamente ou auditar `eoa.get_history(SESSION_ID)` e `store.get_validation_logs(SESSION_ID)`.

In [ ]:
# [TEMPORARIO - APAGAR ESTE BLOCO ANTES DE GERAR O ZIP FINAL]
# Comparativo de ganho ao registrar as 2 ferramentas livres (get_sensor_readings e get_baseline_comparison):

context_agent = eoa.get_agent("context_manager")
context_agent.register_tool(get_sensor_readings)
context_agent.register_tool(get_baseline_comparison)
eoa.inspect()

res_com_novas_tools = eoa.explain(
    incident_id=5,
    session_id=SESSION_ID,
    user_query=(
        "Agora considerando as leituras exatas dos sensores e os desvios em relação à operação 100% saudável, "
        "explique por que a eficiência caiu para 20% no incidente 5 e cite os valores e z-scores dos sensores mais relevantes."
    ),
)

print(">>> ANTES (Parte 1 - apenas load_domain_context + get_incident):")
print(res_predicao.narrative)
print(f"\nNota Validador (antes): {res_predicao.validation_score}/10.0\n")

print(">>> DEPOIS (Parte 2 - com get_sensor_readings + get_baseline_comparison):")
mostrar_resposta(res_com_novas_tools)